Python Algorithmic Tradiing Bot

In [3]:
!pip install yfinance

In [9]:
import yfinance as yf
from datetime import datetime
import time

In [10]:
class TradingStrategy:
    def __init__(self,name):
        self.__name = name

    def generate_signal(self, price_data):
        print("This method should be overridden by subclasses")
        return "hold"
    @property
    def name(self):
        return self.__name

In [11]:
class MySmaTradingStrategy(TradingStrategy):
    def __init__(self, short_window, long_window):
        super().__init__("Simple Moving Average Strategy")
        self.__short_window = short_window
        self.__long_window = long_window

    def generate_signal(self, price_data):
        if len(price_data) < self.long_window:
            return "hold"

        short_avg = sum(price_data[-self.short_window:]) / self.__short_window
        long_avg = sum(price_data[-self.long_window:]) / self.__long_window

        if short_avg > long_avg:
            return "Buy"
        elif short_avg < long_avg:
            return "Sell"
        else:
            return "Hold"
    @property
    def short_window(self):
        return self.__short_window
    @property
    def long_window(self):
        return self.__long_window

In [12]:
class Trade:
    def __init__(self, strategy_name, signal, amount):
        self.__strategy_name = strategy_name
        self.__signal = signal
        self.__amount = amount
        self.__timestamp = datetime.now()

    def execute(self):
        print(f"Executing {self.__signal} for {self.__amount} units using {self.__strategy_name} at {self.__timestamp}")

    @property
    def strategy_name(self):
        return self.__strategy_name

    @property
    def signal(self):
        return self.__signal

    @property
    def amount(self):
        return self.__amount

    @property
    def timestamp(self):
        return self.__timestamp

In [13]:
class MockTradingAPI:
    def __init__(self, balance):
        self.__balance = balance

    def place_order(self, trade, price):
        if trade.signal == "buy" and self.__balance >= trade.amount * price:
            self.__balance -= trade.amount * price
            print(f"Placed buy order for {trade.amount} units at {price}. Remaining balance: {self.__balance}")
        elif trade.signal == "sell":
            self.__balance += trade.amount * price
            print(f"Placed sell order for {trade.amount} units at {price}. Remaining balance: {self.__balance}")
        else:
            print("Insufficient balance or invalid signal.")
    @property
    def get_balance(self):
        return self.__balance

In [20]:
class TradingSystem:
    def __init__(self, api, strategy, symbol):
        self.__api = api
        self.__strategy = strategy
        self.__symbol = symbol
        self.__price_data = []

    def fetch_price_data(self):
        data = yf.download(tickers=self.__symbol, period='1d', interval='1m')
        if not data.empty:
            price = data['Close'].iloc[:, 0].dropna().tolist()
            self.__price_data = price[-self.__strategy.long_window:]  
            if len(self.__price_data) > self.__strategy.long_window:
                self.__price_data.pop(0)
            print(f"Fetched new price data: {price}")
        else:
            print("No data fetched")

    def run(self):
        self.fetch_price_data()
        signal = self.__strategy.generate_signal(self.__price_data)
        print(f"Generated signal: {signal}")
        if signal in ["buy", "sell"]:
            trade = Trade(self.__strategy.name, signal, 1)
            trade.execute()
            self.__api.place_order(trade, self.__price_data[-1])

    @property
    def api(self):
        return self.__api

    @property
    def strategy(self):
        return self.__strategy

    @property
    def symbol(self):
        return self.__symbol

    @property
    def price_data(self):
        return self.__price_data

In [21]:
symbol = 'AAPL'
api = MockTradingAPI(balance=10000)
strategy = MySmaTradingStrategy(short_window=3, long_window=5)
system = TradingSystem(api, strategy, symbol)

for _ in range(10):
    system.run()
    print(f"Remaining balance: {api.get_balance}")
    time.sleep(2)

[*********************100%***********************]  1 of 1 completed


Fetched new price data: [255.2899932861328, 255.17999267578125, 255.3300018310547, 255.5, 255.61500549316406, 255.55499267578125, 255.75999450683594, 255.85000610351562, 255.65660095214844, 255.25990295410156, 255.08999633789062, 255.03500366210938, 255.33999633789062, 255.66720581054688, 255.77000427246094, 256.0299987792969, 256.07000732421875, 256.17999267578125, 256.0201110839844, 256.01251220703125, 255.72210693359375, 255.61000061035156, 255.64999389648438, 255.99000549316406, 256.1199951171875, 256.239990234375, 255.75999450683594, 255.52999877929688, 255.91000366210938, 255.80999755859375, 255.84500122070312, 255.6676025390625, 255.625, 255.5050048828125, 255.49000549316406, 255.4199981689453, 255.57000732421875, 255.47000122070312, 255.35000610351562, 255.22999572753906, 255.25, 255.27000427246094, 255.1999969482422, 255.17999267578125, 255.1999969482422, 255.19000244140625, 255.01150512695312, 254.69500732421875, 254.77000427246094, 254.94000244140625, 254.9250030517578, 255.

[*********************100%***********************]  1 of 1 completed


Fetched new price data: [255.2899932861328, 255.17999267578125, 255.3300018310547, 255.5, 255.61500549316406, 255.55499267578125, 255.75999450683594, 255.85000610351562, 255.65660095214844, 255.25990295410156, 255.08999633789062, 255.03500366210938, 255.33999633789062, 255.66720581054688, 255.77000427246094, 256.0299987792969, 256.07000732421875, 256.17999267578125, 256.0201110839844, 256.01251220703125, 255.72210693359375, 255.61000061035156, 255.64999389648438, 255.99000549316406, 256.1199951171875, 256.239990234375, 255.75999450683594, 255.52999877929688, 255.91000366210938, 255.80999755859375, 255.84500122070312, 255.6676025390625, 255.625, 255.5050048828125, 255.49000549316406, 255.4199981689453, 255.57000732421875, 255.47000122070312, 255.35000610351562, 255.22999572753906, 255.25, 255.27000427246094, 255.1999969482422, 255.17999267578125, 255.1999969482422, 255.19000244140625, 255.01150512695312, 254.69500732421875, 254.77000427246094, 254.94000244140625, 254.9250030517578, 255.

[*********************100%***********************]  1 of 1 completed


...

Fetched new price data: [255.2899932861328, 255.17999267578125, 255.3300018310547, 255.5, 255.61500549316406, 255.55499267578125, 255.75999450683594, 255.85000610351562, 255.65660095214844, 255.25990295410156, 255.08999633789062, 255.03500366210938, 255.33999633789062, 255.66720581054688, 255.77000427246094, 256.0299987792969, 256.07000732421875, 256.17999267578125, 256.0201110839844, 256.01251220703125, 255.72210693359375, 255.61000061035156, 255.64999389648438, 255.99000549316406, 256.1199951171875, 256.239990234375, 255.75999450683594, 255.52999877929688, 255.91000366210938, 255.80999755859375, 255.84500122070312, 255.6676025390625, 255.625, 255.5050048828125, 255.49000549316406, 255.4199981689453, 255.57000732421875, 255.47000122070312, 255.35000610351562, 255.22999572753906, 255.25, 255.27000427246094, 255.1999969482422, 255.17999267578125, 255.1999969482422, 255.19000244140625, 255.01150512695312, 254.69500732421875, 254.77000427246094, 254.94000244140625, 254.9250030517578, 255.

[*********************100%***********************]  1 of 1 completed


Fetched new price data: [255.2899932861328, 255.17999267578125, 255.3300018310547, 255.5, 255.61500549316406, 255.55499267578125, 255.75999450683594, 255.85000610351562, 255.65660095214844, 255.25990295410156, 255.08999633789062, 255.03500366210938, 255.33999633789062, 255.66720581054688, 255.77000427246094, 256.0299987792969, 256.07000732421875, 256.17999267578125, 256.0201110839844, 256.01251220703125, 255.72210693359375, 255.61000061035156, 255.64999389648438, 255.99000549316406, 256.1199951171875, 256.239990234375, 255.75999450683594, 255.52999877929688, 255.91000366210938, 255.80999755859375, 255.84500122070312, 255.6676025390625, 255.625, 255.5050048828125, 255.49000549316406, 255.4199981689453, 255.57000732421875, 255.47000122070312, 255.35000610351562, 255.22999572753906, 255.25, 255.27000427246094, 255.1999969482422, 255.17999267578125, 255.1999969482422, 255.19000244140625, 255.01150512695312, 254.69500732421875, 254.77000427246094, 254.94000244140625, 254.9250030517578, 255.

[*********************100%***********************]  1 of 1 completed


Fetched new price data: [255.2899932861328, 255.17999267578125, 255.3300018310547, 255.5, 255.61500549316406, 255.55499267578125, 255.75999450683594, 255.85000610351562, 255.65660095214844, 255.25990295410156, 255.08999633789062, 255.03500366210938, 255.33999633789062, 255.66720581054688, 255.77000427246094, 256.0299987792969, 256.07000732421875, 256.17999267578125, 256.0201110839844, 256.01251220703125, 255.72210693359375, 255.61000061035156, 255.64999389648438, 255.99000549316406, 256.1199951171875, 256.239990234375, 255.75999450683594, 255.52999877929688, 255.91000366210938, 255.80999755859375, 255.84500122070312, 255.6676025390625, 255.625, 255.5050048828125, 255.49000549316406, 255.4199981689453, 255.57000732421875, 255.47000122070312, 255.35000610351562, 255.22999572753906, 255.25, 255.27000427246094, 255.1999969482422, 255.17999267578125, 255.1999969482422, 255.19000244140625, 255.01150512695312, 254.69500732421875, 254.77000427246094, 254.94000244140625, 254.9250030517578, 255.